# Modelo de clasificación — ¿Llegará el vuelo a tiempo o atrasado?

Cierre de la Unidad 5: predice `ArrDel15` (1 = atraso ≥15 min) usando las features identificadas en el EDA: aerolínea, ruta, mes, día de la semana y franja horaria.

**Nota de alcance:** por el volumen (21M filas), el entrenamiento usa una muestra estratificada representativa — práctica estándar en Machine Learning, distinta de la regla de "no muestrear" que aplica al pipeline analítico de Big Data (Unidades 2-4), donde sí se procesó el 100% de los datos.


In [1]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

con = duckdb.connect()
PARQUET = "data/parquet/**/*.parquet"


## 1. Construir una muestra aleatoria representativa

Debido al volumen del conjunto de datos, se utiliza aproximadamente el 10 % de los registros para el entrenamiento de los modelos de Machine Learning.

La muestra conserva prácticamente la misma distribución de la variable objetivo observada en el conjunto completo. Posteriormente, la división entre entrenamiento y prueba se realiza de manera estratificada para mantener dicha proporción en ambos conjuntos.


In [2]:
# ---------------------------------------------------------
# CREAR UNA MUESTRA ALEATORIA DEL DATASET
# ---------------------------------------------------------

# Usamos aproximadamente el 10 % de los vuelos.
# El dataset completo tiene cerca de 21 millones de registros,
# así que deberíamos obtener alrededor de 2 millones.

PORCENTAJE_MUESTRA = 10

muestra = con.execute(f"""
    SELECT
        ArrDel15,
        Reporting_Airline,
        Origin,
        Dest,
        DayOfWeek,
        Month,

        -- CRSDepTime viene en formato HHMM.
        -- Ejemplo: 1735 significa 17:35.
        -- FLOOR(1735 / 100) devuelve 17.
        FLOOR(CRSDepTime / 100) AS hora_salida,

        Distance

    FROM parquet_scan('{PARQUET}')

    -- Solamente usamos vuelos donde sabemos
    -- si llegaron atrasados o no.
    WHERE ArrDel15 IS NOT NULL

    -- Tomamos aproximadamente el 10 % del dataset.
    USING SAMPLE 10 PERCENT (bernoulli)

""").fetchdf()

print(f"Filas en la muestra: {len(muestra):,}")

print("\nDistribución de la variable objetivo:")
print(
    muestra["ArrDel15"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Filas en la muestra: 2,057,418

Distribución de la variable objetivo:
ArrDel15
0.0    78.75
1.0    21.25
Name: proportion, dtype: float64


In [3]:
# ---------------------------------------------------------
# AGRUPAR AEROPUERTOS POCO FRECUENTES
# ---------------------------------------------------------

# Encontramos los 20 aeropuertos de origen más frecuentes.
top_origenes = muestra["Origin"].value_counts().head(20).index

# Encontramos independientemente
# los 20 aeropuertos de destino más frecuentes.
top_destinos = muestra["Dest"].value_counts().head(20).index


# Todo aeropuerto fuera del top 20 se agrupa como "Otro".
# Esto evita crear cientos de columnas después
# cuando hagamos One-Hot Encoding.

muestra["Origin"] = muestra["Origin"].where(
    muestra["Origin"].isin(top_origenes),
    "Otro"
)

muestra["Dest"] = muestra["Dest"].where(
    muestra["Dest"].isin(top_destinos),
    "Otro"
)


print("Principales aeropuertos de origen:")
print(muestra["Origin"].value_counts().head())

print("\nPrincipales aeropuertos de destino:")
print(muestra["Dest"].value_counts().head())

Principales aeropuertos de origen:
Origin
Otro    988474
ATL      97954
DEN      89687
DFW      89004
ORD      85363
Name: count, dtype: int64

Principales aeropuertos de destino:
Dest
Otro    987838
ATL      97861
DEN      89746
DFW      89306
ORD      84772
Name: count, dtype: int64


## 2. Separar features y variable objetivo, train/test

80% entrenamiento / 20% prueba, estratificado para que ambos conjuntos mantengan la misma proporción de atrasos.


In [4]:
X = muestra.drop(columns=["ArrDel15"])
y = muestra["ArrDel15"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Entrenamiento: {len(X_train):,} filas")
print(f"Prueba:        {len(X_test):,} filas")


Entrenamiento: 1,645,934 filas
Prueba:        411,484 filas


## 3. Pipeline de preprocesamiento

Las columnas categóricas (`Reporting_Airline`, `Origin`, `Dest`, `DayOfWeek`, `Month`, `hora_salida`) se codifican con One-Hot. `Distance` es numérica y queda tal cual.

Usamos un `Pipeline` de scikit-learn para que el mismo preprocesamiento se aplique de forma consistente en entrenamiento y prueba (evita errores comunes de "se me olvidó aplicar el mismo encoding").


In [5]:
columnas_categoricas = ["Reporting_Airline", "Origin", "Dest", "DayOfWeek", "Month", "hora_salida"]
columnas_numericas = ["Distance"]

preprocesador = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
], remainder="passthrough")  # Distance pasa sin transformar


## 4. Modelo 1 — Regresión Logística (baseline)

`class_weight="balanced"` le dice al modelo que le preste más atención a la clase minoritaria (atrasados), compensando el desbalance 79/21.


In [6]:
modelo_logistico = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

modelo_logistico.fit(X_train, y_train)
pred_logistico = modelo_logistico.predict(X_test)
proba_logistico = modelo_logistico.predict_proba(X_test)[:, 1]

print("=== Regresión Logística ===")
print(f"Exactitud:  {accuracy_score(y_test, pred_logistico):.3f}")
print(f"Precisión:  {precision_score(y_test, pred_logistico):.3f}")
print(f"Recall:     {recall_score(y_test, pred_logistico):.3f}")
print(f"F1:         {f1_score(y_test, pred_logistico):.3f}")
print(f"AUC:        {roc_auc_score(y_test, proba_logistico):.3f}")


=== Regresión Logística ===
Exactitud:  0.599
Precisión:  0.296
Recall:     0.645
F1:         0.406
AUC:        0.659


## 5. Modelo 2 — Random Forest

`n_estimators=100` es un punto de partida razonable; `n_jobs=-1` usa todos los núcleos disponibles para entrenar más rápido.


In [7]:
modelo_bosque = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", RandomForestClassifier(
        n_estimators=100, max_depth=15, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )),
])

modelo_bosque.fit(X_train, y_train)
pred_bosque = modelo_bosque.predict(X_test)
proba_bosque = modelo_bosque.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(f"Exactitud:  {accuracy_score(y_test, pred_bosque):.3f}")
print(f"Precisión:  {precision_score(y_test, pred_bosque):.3f}")
print(f"Recall:     {recall_score(y_test, pred_bosque):.3f}")
print(f"F1:         {f1_score(y_test, pred_bosque):.3f}")
print(f"AUC:        {roc_auc_score(y_test, proba_bosque):.3f}")


=== Random Forest ===
Exactitud:  0.597
Precisión:  0.296
Recall:     0.651
F1:         0.407
AUC:        0.663


## 6. Comparación final y matriz de confusión

La matriz de confusión muestra, de los vuelos que SÍ se atrasaron, cuántos detectó el modelo (eso es el recall de la clase positiva) — suele ser más informativo que solo mirar la exactitud.


In [8]:
comparacion = pd.DataFrame({
    "modelo": ["Regresión Logística", "Random Forest"],
    "exactitud": [accuracy_score(y_test, pred_logistico), accuracy_score(y_test, pred_bosque)],
    "precision": [precision_score(y_test, pred_logistico), precision_score(y_test, pred_bosque)],
    "recall": [recall_score(y_test, pred_logistico), recall_score(y_test, pred_bosque)],
    "f1": [f1_score(y_test, pred_logistico), f1_score(y_test, pred_bosque)],
    "auc": [roc_auc_score(y_test, proba_logistico), roc_auc_score(y_test, proba_bosque)],
}).round(3)

comparacion


,modelo,exactitud,precision,recall,f1,auc
0,Regresión Logística,0.599,0.296,0.645,0.406,0.659
1,Random Forest,0.597,0.296,0.651,0.407,0.663


In [11]:
print("Matriz de confusión — Random Forest")
print(confusion_matrix(y_test, pred_bosque))
print()
print(classification_report(y_test, pred_bosque, target_names=["A tiempo", "Atrasado"]))


Matriz de confusión — Random Forest
[[188824 135211]
 [ 30527  56922]]

              precision    recall  f1-score   support

    A tiempo       0.86      0.58      0.69    324035
    Atrasado       0.30      0.65      0.41     87449

    accuracy                           0.60    411484
   macro avg       0.58      0.62      0.55    411484
weighted avg       0.74      0.60      0.63    411484



## 7. Variables más importantes (interpretación para el documento)

Random Forest permite ver qué variables pesaron más en la predicción — útil para la narrativa de conclusiones ("el modelo confirma que X es el factor más influyente").


In [10]:
nombres_features = modelo_bosque.named_steps["preprocesamiento"].get_feature_names_out()
importancias = modelo_bosque.named_steps["clasificador"].feature_importances_

top_importancias = (
    pd.DataFrame({"feature": nombres_features, "importancia": importancias})
    .sort_values("importancia", ascending=False)
    .head(15)
)
top_importancias


,feature,importancia
82,cat__hora_salida_6.0,0.129373
83,cat__hora_salida_7.0,0.076628
81,cat__hora_salida_5.0,0.067134
70,cat__Month_7,0.056396
100,remainder__Distance,0.052814
84,cat__hora_salida_8.0,0.049006
95,cat__hora_salida_19.0,0.038064
94,cat__hora_salida_18.0,0.032849
96,cat__hora_salida_20.0,0.028479
69,cat__Month_6,0.028016


## Interpretación de los resultados

Los dos modelos presentan un desempeño similar. La Regresión Logística alcanzó un AUC de 0.659 y un recall de 0.645, mientras que Random Forest obtuvo un AUC de 0.663 y un recall de 0.651.

Random Forest presentó una mejora ligera en la capacidad de identificar vuelos atrasados, detectando aproximadamente el 65 % de los casos positivos. Sin embargo, la precisión de la clase atrasada fue de aproximadamente 30 %, lo que indica una cantidad considerable de falsos positivos.

Las variables de mayor importancia se relacionan principalmente con la hora programada de salida, seguida por el mes y la distancia del vuelo. Esto es consistente con los patrones identificados previamente durante el análisis exploratorio.

Estos resultados indican que las variables operacionales disponibles antes del vuelo contienen capacidad predictiva, pero no explican completamente los atrasos. Variables adicionales como clima en tiempo real, congestión aeroportuaria y tráfico aéreo podrían mejorar el desempeño del modelo.

**Guardar el modelo final** (opcional, si quieren dejarlo listo para usar sin reentrenar):
```python
import joblib
joblib.dump(modelo_bosque, "data/resultados/modelo_atrasos.pkl")
```


In [12]:
# ---------------------------------------------------------
# GUARDAR RESULTADOS DEL MODELO
# ---------------------------------------------------------

from pathlib import Path

# Crear carpeta de resultados si no existe
CARPETA_RESULTADOS = Path("data/resultados")
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# 1. Guardar tabla de métricas de los modelos
ruta_metricas = CARPETA_RESULTADOS / "metricas_modelos.csv"
comparacion.to_csv(ruta_metricas, index=False)

# 2. Guardar variables más importantes del Random Forest
ruta_importancias = CARPETA_RESULTADOS / "importancia_variables.csv"
top_importancias.to_csv(ruta_importancias, index=False)

print("Resultados guardados correctamente:")
print(f"- {ruta_metricas}")
print(f"- {ruta_importancias}")

Resultados guardados correctamente:
- data\resultados\metricas_modelos.csv
- data\resultados\importancia_variables.csv


In [13]:
# ---------------------------------------------------------
# GUARDAR EL MODELO RANDOM FOREST
# ---------------------------------------------------------

import joblib

ruta_modelo = CARPETA_RESULTADOS / "modelo_atrasos.pkl"

joblib.dump(
    modelo_bosque,
    ruta_modelo
)

print(f"Modelo guardado correctamente en: {ruta_modelo}")

Modelo guardado correctamente en: data\resultados\modelo_atrasos.pkl
